In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("data/twic_data.duckdb")

In [ ]:
df = conn.execute("SELECT * FROM twic_games").df()
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
#df_ = df[df['White'] == 'Grischuk,A']
#df_

In [ ]:
query = """
    WITH white_stats AS (
        SELECT 
            White AS Player,
            COUNT(*) AS total_white_games,
            
            -- Win Rate Example (White wins if '1-0')
            AVG(CASE WHEN Result = '1-0' THEN 1.0 ELSE 0.0 END) AS white_win_rate,
            
            -- Average Moves Example (Counting the periods)
            AVG(LENGTH(Moves) - LENGTH(REPLACE(Moves, '.', ''))) AS white_avg_moves,
            
            -- [YOUR TURN: Add White Draw Rate...]
            AVG(CASE WHEN Result = '1/2-1/2' THEN 1.0 ELSE 0.0 END) AS white_draw_rate,
            -- [YOUR TURN: Add White Captures ('x')...]
            AVG(LENGTH(Moves) - LENGTH(REPLACE(Moves, 'x', ''))) AS white_avg_captures,
            -- [YOUR TURN: Add White Checks ('+')...]
            AVG(LENGTH(Moves) - LENGTH(REPLACE(Moves, '+', ''))) AS white_avg_checks,
            -- [YOUR TURN: Add White Unique Openings (COUNT DISTINCT)...]
            COUNT(DISTINCT ECO) AS white_unique_openings,
            -- [YOUR TURN: Add White Avg Opponent Elo...]
            AVG(BlackElo) AS white_avg_opp_elo
        FROM twic_games
        GROUP BY White
    ),
    
    black_stats AS (
        SELECT 
            Black AS Player,
            COUNT(*) AS total_black_games,
            
            -- Hint: Black wins when the result is '0-1'!
            AVG(CASE WHEN Result = '0-1' THEN 1.0 ELSE 0.0 END) AS black_win_rate,
            
            -- Average Moves Example (Counting the periods)
            AVG(LENGTH(Moves) - LENGTH(REPLACE(Moves, '.', ''))) AS black_avg_moves,
            
            -- [YOUR TURN: Add White Draw Rate...]
            AVG(CASE WHEN Result = '1/2-1/2' THEN 1.0 ELSE 0.0 END) AS black_draw_rate,
            -- [YOUR TURN: Add White Captures ('x')...]
            AVG(LENGTH(Moves) - LENGTH(REPLACE(Moves, 'x', ''))) AS black_avg_captures,
            -- [YOUR TURN: Add White Checks ('+')...]
            AVG(LENGTH(Moves) - LENGTH(REPLACE(Moves, '+', ''))) AS black_avg_checks,
            -- [YOUR TURN: Add White Unique Openings (COUNT DISTINCT)...]
            COUNT(DISTINCT ECO) AS black_unique_openings,
            -- [YOUR TURN: Add White Avg Opponent Elo...]
            AVG(WhiteElo) AS black_avg_opp_elo
        FROM twic_games
        GROUP BY Black
    )
    
    SELECT 
        w.Player,
        (w.total_white_games + b.total_black_games) AS total_games,
        w.white_win_rate,
        b.black_win_rate,
        w.white_avg_moves,
        b.black_avg_moves,
        w.white_draw_rate,
        b.black_draw_rate,
        w.white_avg_captures,
        b.black_avg_captures,
        w.white_avg_checks,
        b.black_avg_checks,
        w.white_unique_openings,
        b.black_unique_openings,
        w.white_avg_opp_elo,
        b.black_avg_opp_elo        
    FROM white_stats w
    JOIN black_stats b ON w.Player = b.Player
    
    -- Filter out the noise: Only keep players with a meaningful sample size
    WHERE (w.total_white_games + b.total_black_games) > 100
"""

df = conn.execute(query).df()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
""" from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from typing import List
import matplotlib.pyplot as plt

# We will store the results of our loop here
inertias: List[float] = []
silhouette_scores: List[float] = []
k_range = range(2, 11) # Test 2 through 10 clusters

for k in k_range:
    # 1. Initialize KMeans with 'k' clusters (use random_state=42 for consistency)
    kmeans = KMeans(n_clusters=k, random_state=42)
    # 2. Fit the model on your 'scaled_features'
    kmeans.fit(scaled_features)
    # 3. Append the model's inertia (kmeans.inertia_) to the inertias list
    inertias.append(kmeans.inertia_)
    # 4. Calculate the silhouette score and append it 
    silhouette_scores.append(silhouette_score(scaled_features, kmeans.labels_))
    pass """

In [ ]:
""" # Create a figure with 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# [YOUR TURN: Plot k_range vs inertias on ax1]
# Hint: ax1.plot(x, y, marker='o')
ax1.set_title("Elbow Method (Inertia)")
ax1.plot(k_range, inertias, marker='o')

# [YOUR TURN: Plot k_range vs silhouette_scores on ax2]
ax2.set_title("Silhouette Score")
ax2.plot(k_range, silhouette_scores, marker='o')

plt.show() """

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Isolate Features
# Create 'features_df' by dropping 'Player' and 'total_games' from your main 'df'
features_df = df.drop(['Player', 'total_games'], axis=1)

# 2. Scale the Data
scaler = StandardScaler()
# Use scaler.fit_transform() on features_df and save it as 'scaled_features'
scaled_features = scaler.fit_transform(features_df)


# 3. PCA (Reduce to 2 Dimensions for plotting)
pca = PCA(n_components=2)
# Use pca.fit_transform() on scaled_features and save as 'pca_result'
pca_result = pca.fit_transform(scaled_features)


# 4. K-Means Clustering
# Let's start by guessing there are 4 main "styles" of chess players
kmeans = KMeans(n_clusters=4, random_state=42)
# Use kmeans.fit_predict() on scaled_features and save the cluster labels
cluster_labels = kmeans.fit_predict(scaled_features)


# 5. Bring it all back together
# Add the PCA results (X and Y coordinates) and the Cluster Labels back into your original 'df'
df['pca_x'] = pca_result[:, 0]
df['pca_y'] = pca_result[:, 1]
df['cluster'] = cluster_labels

In [ ]:
plt.figure(figsize=(10, 10))
sns.scatterplot(data=df, x='pca_x', y='pca_y', hue='cluster', palette='viridis', s=15, alpha=0.6)
plt.title("The Map of Grandmaster Styles")
plt.show()

In [ ]:
# Look at the defining characteristics of each cluster
cluster_profiles = df.drop(columns=['pca_x', 'pca_y', 'Player']).groupby('cluster').mean()
cluster_profiles

In [ ]:
import joblib

# Create an outputs directory inside your twic folder if it doesn't exist
joblib.dump(scaler, 'outputs/scaler.joblib')
joblib.dump(pca, 'outputs/pca.joblib')
joblib.dump(kmeans, 'outputs/kmeans.joblib')

In [ ]:
from sklearn.neighbors import NearestNeighbors
import joblib

# 1. Train the KNN Model (Find the 3 closest neighbors)
knn = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn.fit(scaled_features)

# Save the KNN model
joblib.dump(knn, 'outputs/knn.joblib')

# 2. Export the GM Map Data (Streamlit needs this to draw the background)
# We only save the columns we need for the plot to keep the file tiny
df[['Player', 'cluster', 'pca_x', 'pca_y']].to_parquet('outputs/gm_reference.parquet')

print("KNN and Map Data exported successfully!")